# Bài thực hành Deep Learning: Long Short-Term Memory - LSTM

Notebook này triển khai LSTM cho nhận dạng ảnh và sinh văn bản. Do giáo viên không cung cấp dataset nên bài thực hành sử dụng dataset public tương đương.

## Mục tiêu bài thực hành

- Cài đặt LSTM nhận dạng CIFAR10, Cat/Dog, Fashion-MNIST và Nam/Nữ.
- Cài đặt LSTM sinh văn bản từ Truyện Kiều và Twitter.
- Biết cách chuẩn hóa dữ liệu, chia train/validation, train, đánh giá và lưu model.
- Triển khai dự đoán bằng Flask Web.

## Giới thiệu LSTM

LSTM là một biến thể của RNN, được thiết kế để học dữ liệu chuỗi tốt hơn RNN cơ bản. LSTM có trạng thái nhớ dài hạn, giúp mô hình giữ lại thông tin quan trọng qua nhiều time step.

## Forget Gate, Input Gate, Output Gate

- **Forget Gate** quyết định thông tin nào trong bộ nhớ cũ cần quên.
- **Input Gate** quyết định thông tin mới nào sẽ được ghi vào bộ nhớ.
- **Output Gate** quyết định phần thông tin nào được đưa ra làm output tại time step hiện tại.

Nhờ các cổng này, LSTM phù hợp với dữ liệu chuỗi như văn bản, chuỗi thời gian. Với ảnh, ta có thể xem mỗi hàng ảnh là một time step.

In [ ]:
# Import thư viện
from pathlib import Path
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf

from tensorflow.keras.datasets import cifar10, fashion_mnist

from utils.image_preprocess import (
    ensure_project_dirs,
    cifar10_to_sequence,
    fashion_mnist_to_sequence,
    show_sample_images,
)
from utils.labels import CIFAR10_LABELS, FASHION_MNIST_LABELS
from utils.prediction import predict_image_lstm, predict_text_lstm

ensure_project_dirs()
EPOCHS = 3

## Hàm dùng chung cho xử lý ảnh

Các hàm chính nằm trong `utils/image_preprocess.py`. Ảnh được chuẩn hóa pixel về `[0, 1]` và chuyển sang dạng sequence cho LSTM.

In [ ]:
# Ví dụ shape đầu vào cho LSTM ảnh
print('CIFAR10:', (32, 96))
print('Fashion-MNIST:', (28, 28))
print('Cat/Dog, Nam/Nữ:', (64, 192))

## Hàm dùng chung cho xử lý văn bản

Các hàm chính nằm trong `utils/text_preprocess.py`: làm sạch text, tạo n-gram sequence, padding, lưu tokenizer và sinh văn bản.

In [ ]:
from utils.text_preprocess import clean_vietnamese_text, clean_twitter_text

print(clean_vietnamese_text('Trăm năm trong cõi người ta!!!'))
print(clean_twitter_text('@user I love #DeepLearning https://example.com'))

## Bài 1: LSTM nhận dạng CIFAR10

Dataset CIFAR10 được load bằng `tensorflow.keras.datasets.cifar10`. Ảnh gốc `32x32x3`, sau chuẩn hóa được reshape thành `(samples, 32, 96)`.

In [ ]:
(x_train_cifar, y_train_cifar), (x_test_cifar, y_test_cifar) = cifar10.load_data()
show_sample_images(x_train_cifar, y_train_cifar, CIFAR10_LABELS, n=9)
x_train_cifar_seq = cifar10_to_sequence(x_train_cifar[:100])
print('Shape sequence CIFAR10:', x_train_cifar_seq.shape)

In [ ]:
# Train đầy đủ bài CIFAR10 và lưu models/lstm_cifar10.h5
%run training/train_cifar10_lstm.py

## Bài 2: LSTM nhận dạng Cat/Dog

Ưu tiên dùng TensorFlow Datasets `cats_vs_dogs`. Nếu không tải được, đặt ảnh vào `datasets/catdog/train/cat`, `datasets/catdog/train/dog`, `datasets/catdog/val/cat`, `datasets/catdog/val/dog`.

In [ ]:
# Train Cat/Dog và lưu models/lstm_catdog.h5
%run training/train_catdog_lstm.py

## Bài 3: LSTM nhận dạng Fashion-MNIST

Dataset Fashion-MNIST được load bằng `tensorflow.keras.datasets.fashion_mnist`. Ảnh gốc `28x28`, input LSTM là `(samples, 28, 28)`.

In [ ]:
(x_train_fashion, y_train_fashion), (x_test_fashion, y_test_fashion) = fashion_mnist.load_data()
show_sample_images(x_train_fashion, y_train_fashion, FASHION_MNIST_LABELS, n=9)
x_train_fashion_seq = fashion_mnist_to_sequence(x_train_fashion[:100])
print('Shape sequence Fashion-MNIST:', x_train_fashion_seq.shape)

In [ ]:
# Train Fashion-MNIST và lưu models/lstm_fashion_mnist.h5
%run training/train_fashion_mnist_lstm.py

## Bài 4: LSTM nhận dạng khuôn mặt Nam/Nữ

Đặt dữ liệu vào `datasets/gender/train/male`, `datasets/gender/train/female`, `datasets/gender/val/male`, `datasets/gender/val/female`. Nếu chưa có dữ liệu, script sẽ in hướng dẫn và dừng nhẹ nhàng.

In [ ]:
# Train Nam/Nữ và lưu models/lstm_gender.h5
%run training/train_gender_lstm.py

## Bài 5: Sinh văn bản Truyện Kiều

Đọc file `datasets/truyen_kieu.txt`, làm sạch dữ liệu, dùng Tokenizer, tạo n-gram sequence và train LSTM dự đoán từ tiếp theo.

In [ ]:
truyen_kieu_path = Path('datasets/truyen_kieu.txt')
if truyen_kieu_path.exists():
    print('\n'.join(truyen_kieu_path.read_text(encoding='utf-8').splitlines()[:5]))
else:
    print('Chưa có file datasets/truyen_kieu.txt')

In [ ]:
# Train sinh văn bản Truyện Kiều và lưu model/tokenizer
%run training/train_truyen_kieu_lstm.py

## Bài 6: Sinh văn bản Twitter

Đọc file `datasets/twitter_data.csv`, dùng cột `twitter_content`. Nếu dataset public có cột `text`, `content`, `tweet` hoặc `selected_text`, script sẽ tự đổi tên khi đọc.

In [ ]:
twitter_path = Path('datasets/twitter_data.csv')
if twitter_path.exists():
    display(pd.read_csv(twitter_path).head())
else:
    print('Chưa có file datasets/twitter_data.csv')

In [ ]:
# Train sinh văn bản Twitter và lưu model/tokenizer
%run training/train_twitter_lstm.py

## Lưu mô hình

Các script train tự lưu model vào thư mục `models/` và tokenizer vào `tokenizers/`.

In [ ]:
for path in Path('models').glob('*.h5'):
    print(path)
for path in Path('tokenizers').glob('*.pkl'):
    print(path)

## Dự đoán ảnh mới

Hàm `predict_image_lstm(model_path, image_path, model_type)` load model, đọc ảnh, resize, chuẩn hóa, chuyển ảnh sang sequence, dự đoán nhãn, in độ tin cậy và hiển thị ảnh bằng matplotlib.

In [ ]:
# Ví dụ sau khi đã train model:
# predict_image_lstm('models/lstm_fashion_mnist.h5', 'static/uploads/sample.png', 'fashion_mnist')

## Sinh văn bản mới

Sau khi train model sinh văn bản, dùng `predict_text_lstm` để sinh câu mới. Với Twitter, kết quả được đảm bảo có dấu chấm cuối câu nếu câu chưa có dấu kết thúc.

In [ ]:
# Ví dụ sau khi đã train model:
# predict_text_lstm('models/lstm_truyen_kieu.h5', 'tokenizers/tokenizer_truyen_kieu.pkl', 'tram nam', 20)
# predict_text_lstm('models/lstm_twitter.h5', 'tokenizers/tokenizer_twitter.pkl', 'today is', 20, end_with_period=True)

## Bảng tổng kết

| Tên bài | Dataset | Dạng dữ liệu | Input shape | Output | Loss function | File model |
|---|---|---|---|---|---|---|
| CIFAR10 | TensorFlow/Keras CIFAR10 | Ảnh màu | `(32, 96)` | 10 lớp softmax | sparse_categorical_crossentropy | `models/lstm_cifar10.h5` |
| Cat/Dog | TensorFlow Datasets hoặc thư mục ảnh | Ảnh màu | `(64, 192)` | 1 sigmoid | binary_crossentropy | `models/lstm_catdog.h5` |
| Fashion-MNIST | TensorFlow/Keras Fashion-MNIST | Ảnh xám | `(28, 28)` | 10 lớp softmax | sparse_categorical_crossentropy | `models/lstm_fashion_mnist.h5` |
| Nam/Nữ | FairFace hoặc dataset tự chuẩn bị | Ảnh màu | `(64, 192)` | 1 sigmoid | binary_crossentropy | `models/lstm_gender.h5` |
| Truyện Kiều | File text public | Văn bản | `max_sequence_len - 1` | softmax từ vựng | categorical_crossentropy | `models/lstm_truyen_kieu.h5` |
| Twitter | Sentiment140 hoặc Twitter dataset | Văn bản | `max_sequence_len - 1` | softmax từ vựng | categorical_crossentropy | `models/lstm_twitter.h5` |

## Tổng kết

- LSTM phù hợp với dữ liệu chuỗi.
- Với ảnh, LSTM có thể xem mỗi hàng ảnh là một time step.
- CIFAR10 khó hơn Fashion-MNIST vì ảnh màu và nhiều đối tượng.
- Cat/Dog và Nam/Nữ phụ thuộc nhiều vào chất lượng dataset.
- Sinh văn bản cần nhiều dữ liệu và nhiều epochs hơn để câu tự nhiên.
- Twitter có nhiều ký tự nhiễu nên cần làm sạch dữ liệu.

## Kết luận

Bài thực hành đã xây dựng đủ pipeline LSTM: chuẩn bị dữ liệu, chuyển dữ liệu về dạng chuỗi, xây dựng model, train, đánh giá, lưu model, dự đoán ảnh mới, sinh văn bản mới và triển khai Flask Web.